# Control de lectura 3
## Verificación numérica de gradientes mediante diferencias finitas
**20 minutos · Individual · Notebook principal**

Compare dos gradientes candidatos con una aproximación numérica. No necesita derivar
a mano, usar PyTorch ni entrenar una neurona. La lectura previa se realiza antes del control.
Complete una sola expresión, conserve su predicción inicial y justifique con sus resultados.

**Nombre:** Alejandro Chavez

## Forma de trabajo
1. Responda P0 antes de ejecutar (3 minutos).
2. Complete `diferencia_central` (5 minutos).
3. Ejecute y observe las dos tablas (5 minutos).
4. Responda P1 a P3 (5 minutos).
5. Guarde y entregue (2 minutos).

Modifique solo nombre, respuestas y la expresión pendiente. Use `Shift + Enter`.
No borre celdas. El mensaje PENDIENTE indica que falta su implementación, no es una solución.
Si cambia la función, reejecute las celdas desde arriba. No redondee cálculos ni cambie
el punto o los pasos. La notación `1e-5` significa diez elevado a menos cinco.

P0. Antes de ejecutar: ¿qué dos cantidades se comparan al verificar un gradiente? ¿Un h menor siempre mejora la aproximación? Justifique y conserve su predicción.



**1. ¿Qué dos cantidades se comparan?** Dos números calculados en el mismo punto $w$: la **derivada aproximada numéricamente** a partir de la función $f$ (con diferencias centrales) y el **valor del gradiente candidato**, $g_A(w)$ o $g_B(w)$. La aproximación numérica es

$$g_{num} = \frac{f(w+h) - f(w-h)}{2h}$$

y para saber si coincide con el candidato se mide la **discrepancia absoluta** $\lvert g_{num} - g_{candidato} \rvert$. El candidato correcto es el que da una discrepancia pequeña.

**2. ¿Un h menor siempre mejora la aproximación?** Mi predicción es que **no**. Hay dos errores que se mueven en direcciones opuestas:

- **Error de truncamiento** (viene de la fórmula): la fórmula es una aproximación y mejora al achicar $h$. En la diferencia central baja como $h^2$.
- **Error de redondeo** (viene del computador): el numerador resta dos números casi iguales y el resultado se divide por un $h$ muy chico. Cualquier error diminuto en $f(w+h)$ o $f(w-h)$ se amplifica. Este error **empeora** al achicar $h$.

Por eso debe existir un $h$ intermedio que funcione mejor. En doble precisión suele estar cerca de $10^{-5}$.

**3. Lo que espero ver en los tres pasos:**

- $h = 0.1$: aproximación cercana al valor correcto, pero con un error visible.
- $h = 10^{-5}$: la mejor aproximación.
- $h = 10^{-16}$: falla. Un float de doble precisión no puede distinguir $2$ de $2 + 10^{-16}$, porque los números representables cerca de 2 están separados unos $4.4 \times 10^{-16}$. Entonces $w+h$ y $w-h$ se guardarán como $2.0$ exacto, el numerador dará 0 y la aproximación será 0 o un valor sin sentido.


## Función y candidatos proporcionados
$$f(w)=w^3-2w,\qquad g_A(w)=3w^2-2,\qquad g_B(w)=3w-2.$$
Compruebe en **w = 2** usando diferencias centrales con los tres pasos proporcionados.
El objetivo es obtener evidencia independiente de los candidatos: no llame a sus
funciones para calcular la aproximación numérica ni devuelva un valor fijo.

In [1]:
import csv
import math
import sys
from pathlib import Path

# Funciones y configuración proporcionadas. No modificar.
def funcion(w):
    return w**3 - 2*w

def gradiente_A(w):
    return 3*w**2 - 2

def gradiente_B(w):
    return 3*w - 2

w = 2.0
pasos = (1e-1, 1e-5, 1e-16)
assert sys.float_info.radix == 2 and sys.float_info.mant_dig == 53, "Se requiere float binario de doble precisión. Consulte al docente."
print("Entorno preparado: float binario de doble precisión, sin bibliotecas externas.")

Entorno preparado: float binario de doble precisión, sin bibliotecas externas.


## Complete su función
Sustituya `...` por una expresión que use la función recibida `f`, el punto `w` y el paso `h`.
Debe devolver la aproximación por diferencias centrales estudiada en la lectura.
Conserve el resto de esta celda: la guarda solo permite mostrar PENDIENTE si falta su expresión.

In [2]:
def diferencia_central(f, w, h):
    aproximacion = (f(w + h) - f(w - h)) / (2 * h)  # Diferencia central: [f(w+h) - f(w-h)] / (2h)
    if aproximacion is Ellipsis:
        return None
    return aproximacion

## Obtenga la evidencia
El código imprime `g_num`, los candidatos y sus discrepancias **absolutas** con la
aproximación: `abs(g_num - g_candidato)`. Estas columnas permiten comparar este caso
de escala fija; no son una regla universal de validación. La segunda tabla muestra
los puntos perturbados y una comparación de igualdad de los valores almacenados.

In [3]:
# Código proporcionado. Usa su función; no la sustituye por una solución.
resultados = []
print(f"{'h':>10} {'g_num':>18} {'g_A':>8} {'g_B':>8} {'discrep_A':>14} {'discrep_B':>14}")
for h in pasos:
    g = diferencia_central(funcion, w, h)
    if g is None:
        print(f"{h:10.0e}  PENDIENTE: complete diferencia_central y reejecute desde arriba.")
        continue
    if not isinstance(g, (float, int)) or not math.isfinite(g):
        raise ValueError("La función debe devolver un número real finito para estos casos.")
    a, b = gradiente_A(w), gradiente_B(w)
    fila = dict(h=h, g_num=g, g_A=a, g_B=b,
                discrep_A=abs(g-a), discrep_B=abs(g-b),
                w_mas_h=w+h, w_menos_h=w-h, puntos_iguales=(w+h == w-h))
    resultados.append(fila)
    print(f"{h:10.0e} {g:18.12g} {a:8.4g} {b:8.4g} {abs(g-a):14.6e} {abs(g-b):14.6e}")

print("\nPuntos almacenados (17 cifras significativas; no se redondean los cálculos):")
print(f"{'h':>10} {'w + h':>23} {'w - h':>23} {'¿iguales?':>11}")
for h in pasos:
    print(f"{h:10.0e} {w+h:23.17g} {w-h:23.17g} {str(w+h == w-h):>11}")

if len(resultados) == len(pasos):
    destino = Path("resultados_control_3.csv")
    with destino.open("w", newline="", encoding="utf-8") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=list(resultados[0]))
        escritor.writeheader()
        escritor.writerows(resultados)
    print("\nResultados guardados. Su existencia no certifica que la función o las respuestas sean correctas.")
else:
    print("\nACTIVIDAD INCOMPLETA. No se ha generado un CSV nuevo; un archivo anterior no acredita este intento.")

         h              g_num      g_A      g_B      discrep_A      discrep_B
     1e-01              10.01       10        4   1.000000e-02   6.010000e+00
     1e-05      10.0000000002       10        4   1.987388e-10   6.000000e+00
     1e-16                  0       10        4   1.000000e+01   4.000000e+00

Puntos almacenados (17 cifras significativas; no se redondean los cálculos):
         h                   w + h                   w - h   ¿iguales?
     1e-01      2.1000000000000001      1.8999999999999999       False
     1e-05      2.0000100000000001      1.9999899999999999       False
     1e-16                       2                       2        True

Resultados guardados. Su existencia no certifica que la función o las respuestas sean correctas.


P1. ¿Qué candidato respalda una fila fiable? Cite h, la aproximación y ambas discrepancias para justificar su elección.

**Respuesta.** La fila fiable es la de $h = 10^{-5}$:

- Aproximación: $g_{num} = 10.000000000198739$
- Candidato A: $g_A(2) = 10$, discrepancia $\lvert g_{num} - g_A \rvert = 1.987388 \times 10^{-10}$
- Candidato B: $g_B(2) = 4$, discrepancia $\lvert g_{num} - g_B \rvert = 6.000000$

La aproximación coincide con $g_A$ en unas diez cifras y queda a seis unidades de $g_B$. **El candidato compatible con la evidencia es $g_A$.**

**Las otras dos filas:**

- $h = 0.1$ apunta a lo mismo: $g_{num} = 10.010000000000007$, discrepancia con A de $1.0 \times 10^{-2}$ y con B de $6.01$. La discrepancia con A es mayor que en la fila anterior porque el paso es grande (error de truncamiento $\approx h^2 = 0.01$), pero sigue siendo unas 600 veces menor que la discrepancia con B.
- $h = 10^{-16}$ **no sirve para decidir**: $g_{num} = 0.0$, discrepancia con A de $10$ y con B de $4$. Si se mirara solo esta fila se elegiría B, que es el candidato equivocado. Por eso la decisión se toma con las filas donde el método funciona, no con la de menor $h$.

P2. Compare el menor h con el intermedio: aproximación, w + h y w - h. Explique la causa del cambio y contraste con P0.

**Respuesta.** Valores observados (17 cifras, sin redondear):

| h | g_num | w + h | w − h | ¿iguales? |
|:---:|:---:|:---:|:---:|:---:|
| 1e-5 | 10.000000000198739 | 2.0000100000000001 | 1.9999899999999999 | False |
| 1e-16 | 0.0 | 2 | 2 | True |

**Qué cambió.** Con $h = 10^{-5}$ los dos puntos perturbados son distintos y la aproximación da prácticamente 10. Con $h = 10^{-16}$ los dos puntos perturbados son exactamente $2.0$ (la columna de igualdad dice `True`) y la aproximación cae a $0.0$.

**Causa.** Un float de doble precisión guarda 53 bits de mantisa. Cerca de 2, los números representables están separados por $2^{-51} \approx 4.44 \times 10^{-16}$. El paso $10^{-16}$ es menor que la mitad de esa separación, así que al calcular $2.0 + 10^{-16}$ el resultado se redondea a $2.0$, y lo mismo pasa con $2.0 - 10^{-16}$. La perturbación se pierde **antes** de evaluar $f$. Después:

$$g_{num} = \frac{f(2.0) - f(2.0)}{2 \cdot 10^{-16}} = \frac{4.0 - 4.0}{2 \cdot 10^{-16}} = 0.0$$

No es que la fórmula "converja" a 0: es que el computador perdió el paso $h$ por redondeo.

Con $h = 10^{-5}$ el paso sí sobrevive: $2.00001$ y $1.99999$ se distinguen sin problema. En esa fila el error de truncamiento ($\approx h^2 = 10^{-10}$) y el de redondeo ($\approx \varepsilon \cdot \lvert f(2) \rvert / h \approx 2.2 \times 10^{-16} \cdot 4 / 10^{-5} \approx 10^{-10}$) son del mismo tamaño. Por eso la discrepancia con A queda en $\approx 2 \times 10^{-10}$, que es casi lo mejor que puede dar este método.

**Contraste con P0.** La predicción se cumple: bajar $h$ ayuda hasta cierto punto (de $0.1$ a $10^{-5}$ la discrepancia con A baja de $10^{-2}$ a $2 \times 10^{-10}$) y después empeora (de $10^{-5}$ a $10^{-16}$ sube de $2 \times 10^{-10}$ a $10$, el peor resultado posible). El único matiz es que el fallo en $10^{-16}$ fue total y silencioso: un $0.0$ exacto, sin mensaje de error ni aviso.

P3. ¿Coincidir en w = 2 demuestra corrección para cualquier entrada? Justifique y proponga otra comprobación, sin programarla.

**Respuesta: no.** Coincidir en $w = 2$ es una sola comparación. Muchas funciones distintas valen 10 en $w = 2$, por ejemplo $5w$, $w^2 + 6$ o la constante $10$. Todas pasarían esta prueba y todas son incorrectas.

- La evidencia en $w = 2$ **refuta** a $g_B$: da 4 donde la función indica 10.
- A $g_A$ no la demuestra, solo **no la refuta**. Es compatible con la evidencia, nada más.

Además, $g_A$ y $g_B$ valen lo mismo en $w = 0$ (ambas dan $-2$) y en $w = 1$ (ambas dan $1$). Si el control se hubiera hecho en uno de esos puntos, la diferencia central no habría podido distinguirlas.

**Otra comprobación (sin programarla).** Repetir la diferencia central con $h = 10^{-5}$ en varios puntos distintos, por ejemplo $w = -1.5,\ -0.5,\ 0.5,\ 1.5$ y $3$. Así se cubren valores negativos, valores cerca de cero, la zona donde el gradiente cambia de signo y valores grandes. Conviene evitar $w = 0$ y $w = 1$, donde los dos candidatos coinciden.

En cada punto se exige que la discrepancia sea pequeña, medida en forma **relativa** para que la tolerancia no dependa del tamaño del número:

$$\frac{\lvert g_{num} - g_{candidato} \rvert}{\max(1, \lvert g_{candidato} \rvert)} < 10^{-6}$$

Un candidato correcto debe pasar en **todos** los puntos. Basta un punto que falle para descartarlo.

## Entrega y evaluación
Reinicie el kernel, ejecute todo y guarde. Entregue este notebook con su nombre, función,
tablas y respuestas P0 a P3, junto con `resultados_control_3.csv`.
Si usa la alternativa `.py`, entregue el script con las respuestas y el CSV.
Si no termina, conserve su trabajo parcial e indique qué falta.

Se evalúan implementación (30%), evidencia (30%) e interpretación (40%). Una predicción
inicialmente incorrecta puede recibir crédito si está razonada y se contrasta honestamente.
El programa no califica sus respuestas ni verifica el orden en que las escribió.
Los resultados y las respuestas deben corresponder a esta ejecución, no a un CSV anterior.